In [1]:
!pip install gdown  # Ensure gdown is installed

In [3]:
!pip install -q gdown

import gdown
import zipfile

file_id = "1Gqe40iOinVJ8TTKpOGQQIvSxLtgLGDNX"

gdown.download(
    f"https://drive.google.com/uc?id={file_id}",
    "/content/Pothole_Datasets.zip",
    quiet=False
)

with zipfile.ZipFile("/content/Pothole_Datasets.zip", "r") as zip_ref:
    zip_ref.extractall("/content/Pothole_Datasets")

print("Download and extraction completed!")

Downloading...
From (original): https://drive.google.com/uc?id=1Gqe40iOinVJ8TTKpOGQQIvSxLtgLGDNX
From (redirected): https://drive.google.com/uc?id=1Gqe40iOinVJ8TTKpOGQQIvSxLtgLGDNX&confirm=t&uuid=e0571a7c-1369-4519-a64e-384ca16c350f
To: /content/Pothole_Datasets.zip
100%|██████████| 81.3M/81.3M [00:02<00:00, 35.3MB/s]


Download and extraction completed!


In [4]:
import os

for root, dirs, files in os.walk("/content/Pothole_Datasets"):
    print(root)
    print("Folders:", dirs[:5])
    print("Files:", files[:5])
    print("-" * 50)
    break

/content/Pothole_Datasets
Folders: ['Pothole_Datasets']
Files: []
--------------------------------------------------


In [6]:
import os
import shutil
from pathlib import Path
from sklearn.model_selection import train_test_split

# Paths
TRAIN_IMAGES = "/content/Pothole_Datasets/Pothole_Datasets/train/images"
TRAIN_LABELS = "/content/Pothole_Datasets/Pothole_Datasets/train/labels"

TEST_IMAGES = "/content/Pothole_Datasets/Pothole_Datasets/test"

OUTPUT = "/content/final_data"

# Create folders
for split in ["train", "val", "test"]:
    os.makedirs(f"{OUTPUT}/images/{split}", exist_ok=True)

for split in ["train", "val"]:
    os.makedirs(f"{OUTPUT}/labels/{split}", exist_ok=True)

# Collect image-label pairs
image_files = []

for img_path in Path(TRAIN_IMAGES).glob("*"):
    if img_path.suffix.lower() in [".jpg", ".jpeg", ".png", ".bmp", ".webp"]:

        label_path = Path(TRAIN_LABELS) / f"{img_path.stem}.txt"

        if label_path.exists():
            image_files.append(img_path.stem)

print(f"Found {len(image_files)} labelled images")

# Split 90-10
train_ids, val_ids = train_test_split(
    image_files,
    test_size=0.10,
    random_state=42,
    shuffle=True
)

print(f"Train: {len(train_ids)}")
print(f"Val: {len(val_ids)}")

# Copy train
for stem in train_ids:
    for ext in [".jpg", ".jpeg", ".png", ".bmp", ".webp"]:
        img = Path(TRAIN_IMAGES) / f"{stem}{ext}"
        if img.exists():
            shutil.copy2(img, f"{OUTPUT}/images/train/{img.name}")
            break

    shutil.copy2(
        Path(TRAIN_LABELS) / f"{stem}.txt",
        f"{OUTPUT}/labels/train/{stem}.txt"
    )

# Copy val
for stem in val_ids:
    for ext in [".jpg", ".jpeg", ".png", ".bmp", ".webp"]:
        img = Path(TRAIN_IMAGES) / f"{stem}{ext}"
        if img.exists():
            shutil.copy2(img, f"{OUTPUT}/images/val/{img.name}")
            break

    shutil.copy2(
        Path(TRAIN_LABELS) / f"{stem}.txt",
        f"{OUTPUT}/labels/val/{stem}.txt"
    )

# Copy test images only
test_count = 0

for img in Path(TEST_IMAGES).glob("*"):
    if img.suffix.lower() in [".jpg", ".jpeg", ".png", ".bmp", ".webp"]:
        shutil.copy2(img, f"{OUTPUT}/images/test/{img.name}")
        test_count += 1

print(f"Test images copied: {test_count}")
print("Dataset preparation completed.")

Found 985 labelled images
Train: 886
Val: 99
Test images copied: 148
Dataset preparation completed.


In [7]:
yaml_content = """
path: /content/final_data

train: images/train
val: images/val
test: images/test

names:
  0: pothole
"""

with open("/content/final_data/data.yaml", "w") as f:
    f.write(yaml_content)

print("data.yaml created")

data.yaml created


In [8]:
!pip install -q ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.3/41.3 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 31.6 MB/s eta 0:00:00


In [9]:
import torch
print("GPU:", torch.cuda.get_device_name(0))
print("CUDA Available:", torch.cuda.is_available())

GPU: Tesla T4
CUDA Available: True


In [13]:
import ultralytics

print(ultralytics.__file__)
print(ultralytics.__version__)

/usr/local/lib/python3.12/dist-packages/ultralytics/__init__.py
8.4.64


In [14]:
!pip uninstall -y ultralytics
!pip install -U ultralytics


Found existing installation: ultralytics 8.4.64
Uninstalling ultralytics-8.4.64:
  Successfully uninstalled ultralytics-8.4.64
  Using cached ultralytics-8.4.64-py3-none-any.whl.metadata (41 kB)
Using cached ultralytics-8.4.64-py3-none-any.whl (1.3 MB)


In [1]:
from ultralytics import YOLO

model = YOLO("yolov8m.pt")
print("Loaded successfully")

Loaded successfully


In [2]:
from ultralytics import YOLO
import torch

print("GPU:", torch.cuda.get_device_name(0))

model = YOLO("yolov8m.pt")

results = model.train(
    data="/content/final_data/data.yaml",

    epochs=60,
    imgsz=640,

    batch=12,              # Auto-select maximum batch size
    device=0,
    workers=8,

    pretrained=True,
    cache=True,

    optimizer="AdamW",
    lr0=0.001,
    lrf=0.01,
    weight_decay=0.0005,

    cos_lr=True,
    patience=20,

    # Augmentations
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,

    degrees=10,
    translate=0.15,
    scale=0.5,
    shear=2.0,

    fliplr=0.5,
    flipud=0.2,

    mosaic=1.0,
    mixup=0.15,
    copy_paste=0.1,

    amp=True,

    project="/content/drive/MyDrive/pothole_training",
    name="yolov8m_60epochs",
    exist_ok=True
)

GPU: Tesla T4
Ultralytics 8.4.64 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=12, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.1, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/final_data/data.yaml, degrees=10, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=60, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.2, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.15, mode=train, model=yolov8m.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolov8m_60epochs, nbs=64, nms=False, opset=None, optimize=False, optimizer=AdamW, overlap_mas

In [4]:
from pathlib import Path

weights_dir = Path("/content/drive/MyDrive/pothole_training/yolov8m_60epochs/weights")

for f in weights_dir.iterdir():
    print(f.name, round(f.stat().st_size / 1024 / 1024, 2), "MB")

last.pt 49.62 MB
best.pt 49.62 MB


In [5]:
from ultralytics import YOLO

model = YOLO(
    "/content/drive/MyDrive/pothole_training/yolov8m_60epochs/weights/best.pt"
)

model.train(
    data="/content/final_data/data.yaml",

    epochs=50,          # additional training
    imgsz=640,
    batch=12,
    device=0,

    optimizer="AdamW",
    lr0=0.0001,         # lower LR for fine-tuning
    lrf=0.001,

    cache=True,
    workers=8,

    project="/content/drive/MyDrive/pothole_training",
    name="yolov8m_finetune_50epochs",
    exist_ok=True
)

Ultralytics 8.4.64 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=12, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/final_data/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0001, lrf=0.001, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/content/drive/MyDrive/pothole_training/yolov8m_60epochs/weights/best.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolov8m_finetune_50epochs, nbs=64, nms=Fal

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7b3071ea0500>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.048048, 

In [7]:
from ultralytics import YOLO

best_model = YOLO(
    "/content/drive/MyDrive/pothole_training/yolov8m_finetune_50epochs/weights/best.pt"
)

metrics = best_model.val(
    data="/content/final_data/data.yaml"
)

print(metrics)

Ultralytics 8.4.64 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 93 layers, 25,840,339 parameters, 0 gradients, 78.7 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 26.9±13.1 MB/s, size: 67.8 KB)
val: Scanning /content/final_data/labels/val.cache... 99 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 99/99 26.0Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 7/7 1.4it/s 5.0s
                   all         99        371      0.887      0.824      0.907      0.584
Speed: 8.4ms preprocess, 23.8ms inference, 0.0ms loss, 2.3ms postprocess per image
Results saved to /content/runs/detect/val
ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7b3071e1c380>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-C

In [8]:
best_model.predict(
    source="/content/final_data/images/test",
    imgsz=640,
    conf=0.25,
    save=True
)


image 1/148 /content/final_data/images/test/a-closeup-shot-of-a-muddy-puddle-in-a-cracked-road.jpg: 480x640 1 pothole, 79.4ms
image 2/148 /content/final_data/images/test/a-large-pot-hole-filled-with-water-on-an-asphalt-road.jpg: 448x640 1 pothole, 85.7ms
image 3/148 /content/final_data/images/test/a-pothole-on-the-road.jpg: 448x640 1 pothole, 27.0ms
image 4/148 /content/final_data/images/test/a-water-filled-hole-on-an-asphalt-street-it-poses-a-danger-t.jpg: 448x640 2 potholes, 27.0ms
image 5/148 /content/final_data/images/test/asphalt-road-close-up.jpg: 640x448 3 potholes, 75.8ms
image 6/148 /content/final_data/images/test/asphalt-road-with-pothole-filled-with-water-and-asphalt-piec.jpg: 448x640 1 pothole, 28.2ms
image 7/148 /content/final_data/images/test/bad-asphalt-road-pit-on-the-road.jpg: 416x640 2 potholes, 74.1ms
image 8/148 /content/final_data/images/test/bad-bumpy-road-background-with-copy-space-for-text.jpg: 448x640 4 potholes, 27.4ms
image 9/148 /content/final_data/images/t

error: OpenCV(4.13.0) /io/opencv/modules/imgcodecs/src/loadsave.cpp:1310: error: (-215:Assertion failed) !buf.empty() in function 'imdecode_'
